# Vector DB — Part 2: Query

**Prerequisite:** run `vectordb-create.ipynb` first to build the ChromaDB collection.

This notebook:
1. Opens the persisted TechVault collection
2. Encodes a query with the same embedding model
3. Finds the top-2 semantically closest chunks
4. Builds a RAG prompt ready for any LLM

The punchline: searching `"WiFi"` matches `"Internet stipend"` — zero shared keywords.

In [1]:
%pip install -q chromadb sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import chromadb
from sentence_transformers import SentenceTransformer

/Users/ashishb/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/ashishb/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load model and open collection

Must use the **exact same model** that was used to create the embeddings. A different model lives in a different vector space — the similarity scores would be meaningless.

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection("techvault-docs")

print(f"Collection loaded: {collection.count()} chunks indexed")

Collection loaded: 247 chunks indexed


## 2. Encode the query

The question becomes a vector in the same 384-dimensional space as the stored documents. That shared space is what makes the match possible.

In [4]:
query = "How much do I get for my home WiFi?"

# Same model — query lives in the same vector space as the stored chunks
query_embedding = model.encode(query).tolist()

print(f"Query: {query!r}")
print(f"Encoded to a vector of {len(query_embedding)} dimensions")

Query: 'How much do I get for my home WiFi?'
Encoded to a vector of 384 dimensions


## 3. Search the collection

`collection.query` returns the top-N most similar chunks by cosine similarity. No keywords involved.

In [8]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2,
)

print("Top matches:")
for i, (doc, distance) in enumerate(zip(results["documents"][0], results["distances"][0])):
    print(f"\n[{i+1}] L2 distance={distance:.3f} (lower = more similar)")
    print(f"    {doc}")

Top matches:

[1] L2 distance=1.105 (lower = more similar)
    INTERNET STIPEND:
- $50/month for home internet
- No receipts required

[2] L2 distance=1.222 (lower = more similar)
    STARTER PLAN:
- $12/user/month (billed annually)
- 1 TB storage per user
- 30-day version history
- Standard support


**`"WiFi"` → matched `"Internet stipend"` ✅**

Zero shared keywords. Pure semantic match.

---

## 4. Build the RAG prompt

Join the retrieved chunks into a context string, drop it into a prompt template, and hand it to any LLM. The retrieval logic is completely LLM-agnostic.

In [6]:
context = "\n".join(results["documents"][0])

prompt = f"""Answer based only on the context below.
Context: {context}
Question: {query}"""

print(prompt)

Answer based only on the context below.
Context: INTERNET STIPEND:
- $50/month for home internet
- No receipts required
STARTER PLAN:
- $12/user/month (billed annually)
- 1 TB storage per user
- 30-day version history
- Standard support
Question: How much do I get for my home WiFi?


## 5. Plug into any LLM

The retrieval step is done. Hand `prompt` to whichever LLM you're using:

In [7]:
# Swap in any LLM — retrieval logic is LLM-agnostic

# OpenAI
# from openai import OpenAI
# client = OpenAI()
# response = client.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": prompt}]
# )
# print(response.choices[0].message.content)

# Anthropic
# import anthropic
# client = anthropic.Anthropic()
# response = client.messages.create(
#     model="claude-haiku-4-5-20251001",
#     max_tokens=256,
#     messages=[{"role": "user", "content": prompt}]
# )
# print(response.content[0].text)

print("Uncomment one of the blocks above to get an LLM answer.")

Uncomment one of the blocks above to get an LLM answer.


---

## How it worked

| Step | What happened |
|------|---------------|
| Query encoded | `"WiFi"` → 384-dim vector |
| L2 distance | Compared against 247 stored vectors |
| Top match | `"Internet stipend: $50/month"` — small distance, closest match |
| Keyword overlap | Zero — pure semantic match |
| Prompt built | Context injected, ready for any LLM |